In [9]:
from qiskit.quantum_info import Statevector, Operator
from numpy import sqrt

In [10]:
zero, one = Statevector.from_label("0"), Statevector.from_label("1")
zero.tensor(one).draw("latex")

<IPython.core.display.Latex object>

In [11]:
plus = Statevector.from_label("+")
i_state = Statevector([1 / sqrt(2), 1j / sqrt(2)])
psi = plus.tensor(i_state)

psi.draw("latex")

<IPython.core.display.Latex object>

In [12]:
X = Operator([[0, 1], [1, 0]])
I = Operator([[1, 0], [0, 1]])

X.tensor(I)

Operator([[0.+0.j, 0.+0.j, 1.+0.j, 0.+0.j],
          [0.+0.j, 0.+0.j, 0.+0.j, 1.+0.j],
          [1.+0.j, 0.+0.j, 0.+0.j, 0.+0.j],
          [0.+0.j, 1.+0.j, 0.+0.j, 0.+0.j]],
         input_dims=(2, 2), output_dims=(2, 2))


In [13]:
psi.evolve(I ^ X).draw("latex")

<IPython.core.display.Latex object>

In [14]:
CX = Operator([
    [1, 0, 0, 0],
    [0, 1, 0, 0],
    [0, 0, 0, 1],
    [0, 0, 1, 0],
])

psi.evolve(CX).draw("latex")

<IPython.core.display.Latex object>

In [15]:
W = Statevector([0, 1, 1, 0, 1, 0, 0, 0]) / sqrt(3)
W.draw("latex")

<IPython.core.display.Latex object>

In [16]:
result, new_sv = W.measure([0]) # measure qubit 0
print(f"Measured: {result}\nState after measurement:")
new_sv.draw("latex")

Measured: 1
State after measurement:


<IPython.core.display.Latex object>

In [8]:
from qiskit_aer import Aer
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.quantum_info import Statevector, random_statevector
from qiskit.visualization import plot_histogram, plot_state_city
from qiskit.quantum_info import Statevector, Operator, partial_trace, entropy, DensityMatrix
from qiskit.quantum_info.states import state_fidelity
import numpy as np
from scipy.linalg import sqrtm
from numpy import sqrt, pi
import matplotlib.pyplot as plt

In [16]:
def create_and_analyze_bell_states():
    # Define basic operators
    I = Operator([[1, 0], [0, 1]])
    H = Operator([[1/sqrt(2), 1/sqrt(2)], [1/sqrt(2), -1/sqrt(2)]])
    X = Operator([[0, 1], [1, 0]])
    
    # Create CNOT operator
    CNOT = Operator([
        [1, 0, 0, 0],
        [0, 1, 0, 0], 
        [0, 0, 0, 1],
        [0, 0, 1, 0]
    ])
    
    bell_states = {}
    
    # |Φ⁺⟩ = (|00⟩ + |11⟩)/√2
    phi_plus = Statevector.from_label('00')
    phi_plus = phi_plus.evolve(H.tensor(I))  # Apply H to first qubit
    phi_plus = phi_plus.evolve(CNOT)         # Apply CNOT
    bell_states["Φ⁺"] = phi_plus
    
    # |Φ⁻⟩ = (|00⟩ - |11⟩)/√2
    phi_minus = Statevector.from_label('00')
    phi_minus = phi_minus.evolve(H.tensor(I))  # Apply H to first qubit
    phi_minus = phi_minus.evolve(CNOT)         # Apply CNOT
    # Apply Z gate to first qubit to get minus sign
    Z = Operator([[1, 0], [0, -1]])
    phi_minus = phi_minus.evolve(Z.tensor(I))
    bell_states["Φ⁻"] = phi_minus
    
    # |Ψ⁺⟩ = (|01⟩ + |10⟩)/√2
    psi_plus = Statevector.from_label('01')
    psi_plus = psi_plus.evolve(H.tensor(I))  # Apply H to first qubit
    psi_plus = psi_plus.evolve(CNOT)         # Apply CNOT
    bell_states["Ψ⁺"] = psi_plus
    
    # |Ψ⁻⟩ = (|01⟩ - |10⟩)/√2
    psi_minus = Statevector.from_label('01')
    psi_minus = psi_minus.evolve(H.tensor(I))  # Apply H to first qubit
    psi_minus = psi_minus.evolve(CNOT)         # Apply CNOT
    # Apply Z gate to first qubit to get minus sign
    psi_minus = psi_minus.evolve(Z.tensor(I))
    bell_states["Ψ⁻"] = psi_minus
    
    # Analyze entanglement for each Bell state
    print("=== Bell States Entanglement Analysis ===\n")
    
    for name, state in bell_states.items():
        print(f"Bell State {name}:")
        print(f"Statevector: {state}")
        
        # Calculate reduced density matrices
        rho_A = partial_trace(state, [1])  # Trace out qubit 1
        rho_B = partial_trace(state, [0])  # Trace out qubit 0
        
        # Calculate entanglement entropy
        entropy_A = entropy(rho_A)
        entropy_B = entropy(rho_B)
        
        print(f"Entanglement Entropy: {entropy_A:.4f}")
        print(f"Reduced density matrix ρ_A:\n{rho_A.data}")
        
        # Verify maximal entanglement
        if abs(entropy_A - 1.0) < 1e-10:
            print("✓ Maximally entangled")
        else:
            print("✗ Not maximally entangled")
        print("-" * 50)
    
    return bell_states

# Run the analysis
bell_states = create_and_analyze_bell_states()

=== Bell States Entanglement Analysis ===

Bell State Φ⁺:
Statevector: Statevector([0.70710678+0.j, 0.        +0.j, 0.        +0.j,
             0.70710678+0.j],
            dims=(2, 2))
Entanglement Entropy: 1.0000
Reduced density matrix ρ_A:
[[0.5+0.j 0. +0.j]
 [0. +0.j 0.5+0.j]]
✓ Maximally entangled
--------------------------------------------------
Bell State Φ⁻:
Statevector: Statevector([ 0.70710678+0.j,  0.        +0.j,  0.        +0.j,
             -0.70710678+0.j],
            dims=(2, 2))
Entanglement Entropy: 1.0000
Reduced density matrix ρ_A:
[[0.5+0.j 0. +0.j]
 [0. +0.j 0.5+0.j]]
✓ Maximally entangled
--------------------------------------------------
Bell State Ψ⁺:
Statevector: Statevector([0.        +0.j, 0.70710678+0.j, 0.70710678+0.j,
             0.        +0.j],
            dims=(2, 2))
Entanglement Entropy: 1.0000
Reduced density matrix ρ_A:
[[0.5+0.j 0. +0.j]
 [0. +0.j 0.5+0.j]]
✓ Maximally entangled
--------------------------------------------------
Bell State Ψ⁻: